# TimeOperTransformer 기반 시계열 공정 TAT 예측 모델

## 📌 개요
TimeOperTransformer는 시간축(timekey_hr)과 공정 순서(oper_id)를 모두 고려한 3D 입력 구조의 Transformer 기반 예측 모델입니다. 시간 윈도우(기본 24시간) 동안의 공정 데이터를 한 번에 처리하여 각 공정별 TAT을 예측합니다.

**데이터 구조**: `[batch_size, time_window(24), max_oper_per_hour(803), features]`

## 🔧 환경 설정 및 라이브러리 import

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import yaml
import logging
import os
import json
from datetime import datetime
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, Subset

## 📊 유틸리티 함수들

### 로깅 및 설정 함수

In [ ]:
def setup_logging():
    """로깅 설정"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[logging.FileHandler("training.log"), logging.StreamHandler()],
    )
    return logging.getLogger(__name__)


logger = setup_logging()


def load_config(config_path: str) -> dict:
    """YAML 설정 파일 로드"""
    with open(config_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    return config


def split_dataset_by_ratio(
    dataset, train_ratio: float = 0.7, val_ratio: float = 0.15, test_ratio: float = 0.15
):
    """Dataset을 비율에 따라 순서대로 분할"""
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    total_size = len(dataset)
    train_size = int(total_size * train_ratio)
    val_size = int(total_size * val_ratio)
    test_size = total_size - train_size - val_size

    train_indices = list(range(0, train_size))
    val_indices = list(range(train_size, train_size + val_size))
    test_indices = list(range(train_size + val_size, total_size))

    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    test_dataset = Subset(dataset, test_indices)

    logger.info(f"Dataset 분할 완료:")
    logger.info(f"  - 전체: {total_size:,}개")
    logger.info(f"  - 훈련: {len(train_dataset):,}개 ({len(train_dataset)/total_size:.1%})")
    logger.info(f"  - 검증: {len(val_dataset):,}개 ({len(val_dataset)/total_size:.1%})")
    logger.info(f"  - 테스트: {len(test_dataset):,}개 ({len(test_dataset)/total_size:.1%})")

    return train_dataset, val_dataset, test_dataset

### 범주형 데이터 처리기

In [ ]:
class CategoricalProcessor:
    """범주형 변수 처리를 위한 클래스"""

    def __init__(self, embedding_dim=8):
        self.embedding_dim = embedding_dim
        self.label_encoders = {}
        self.vocab_sizes = {}
        self.categorical_columns = []

    def fit(self, df, categorical_columns):
        """범주형 컬럼들의 인코더를 학습"""
        self.categorical_columns = categorical_columns

        for col in categorical_columns:
            unique_values = df[col].astype(str).unique()
            encoder = LabelEncoder()
            encoder.fit(unique_values)

            self.label_encoders[col] = encoder
            self.vocab_sizes[col] = len(encoder.classes_)

        logger.info(f"범주형 변수별 고유값 개수:")
        for col in categorical_columns:
            logger.info(f"  {col}: {self.vocab_sizes[col]}개")

    def transform(self, df):
        """DataFrame의 범주형 컬럼들을 숫자로 변환"""
        df_encoded = df.copy()

        for col in self.categorical_columns:
            df_encoded[col] = self.label_encoders[col].transform(df_encoded[col].astype(str))

        return df_encoded

    def get_embedding_specs(self):
        """임베딩 레이어 생성을 위한 스펙 반환"""
        return {
            col: (vocab_size, self.embedding_dim)
            for col, vocab_size in self.vocab_sizes.items()
        }


## 📈 시계열 데이터셋 클래스

### 메인 데이터셋 클래스

In [ ]:
class TimeSeriesOperDataset(Dataset):
    """
    시간대별 oper 데이터를 위한 Dataset
    3D 구조: [time_window, max_oper_per_hour, features]를 유지
    """

    def __init__(
        self,
        df,
        categorical_columns,
        continuous_columns,
        target_column="y",
        categorical_processor=None,
        time_window=24,
        embedding_dim=8,
        sample_id_col=None,
        global_max_oper=None,  # 전체 데이터셋의 max_oper 값
    ):
        self.df = df.copy()
        self.categorical_columns = categorical_columns
        self.continuous_columns = continuous_columns
        self.target_column = target_column
        self.time_window = time_window
        self.embedding_dim = embedding_dim
        self.sample_id_col = sample_id_col

        # 범주형 데이터 처리기 설정
        if categorical_processor is None:
            self.categorical_processor = CategoricalProcessor(embedding_dim)
            self.categorical_processor.fit(df, categorical_columns)
        else:
            self.categorical_processor = categorical_processor

        # 데이터 전처리 - 안전한 인덱스 처리
        self.df_encoded = self.df.copy()
        if categorical_columns:
            categorical_encoded = self.categorical_processor.transform(self.df[categorical_columns])
            
            # 안전성 검증: 인코딩된 값이 유효 범위 내에 있는지 확인
            for col in categorical_columns:
                max_value = categorical_encoded[col].max()
                vocab_size = len(self.categorical_processor.label_encoders[col].classes_)
                if max_value >= vocab_size:
                    logger.warning(
                        f"열 '{col}'에서 범위 초과 값 발견: max={max_value}, vocab_size={vocab_size}"
                    )
                    # 범위를 벗어난 값을 0(unknown)으로 클램핑
                    categorical_encoded[col] = categorical_encoded[col].clip(0, vocab_size - 1)

            self.df_encoded[categorical_columns] = categorical_encoded

        # 전체 데이터셋의 max_oper를 계산하거나 받아서 사용
        if global_max_oper is None:
            self.global_max_oper = self._calculate_global_max_oper()
        else:
            self.global_max_oper = global_max_oper

        # 시간 기반 샘플 분할
        self._prepare_time_based_samples()

        logger.info(f"Dataset 구성 완료:")
        logger.info(f"  - 총 샘플 수: {len(self.samples)}")
        logger.info(f"  - 시간 윈도우 크기: {time_window}")
        logger.info(f"  - 전체 데이터셋 max_oper_per_hour: {self.global_max_oper}")
        logger.info(f"  - 입력 변수: 범주형 {len(categorical_columns)}개 + 연속형 {len(continuous_columns)}개")
        logger.info(f"  - 출력 변수: {target_column}")
        logger.info(f"  - 구조 정보 보존: timekey_hr + oper_id")

    def _calculate_global_max_oper(self):
        """전체 데이터셋에서 시간당 최대 oper 수를 계산"""
        if self.sample_id_col is not None:
            max_oper_counts = []
            sample_ids = self.df_encoded[self.sample_id_col].unique()
            for sample_id in sample_ids:
                sample_df = self.df_encoded[self.df_encoded[self.sample_id_col] == sample_id]
                if len(sample_df) > 0:
                    oper_counts_per_time = sample_df.groupby("timekey_hr").size()
                    max_oper_counts.append(oper_counts_per_time.max())
        else:
            if len(self.df_encoded) > 0:
                oper_counts_per_time = self.df_encoded.groupby("timekey_hr").size()
                max_oper_counts = [oper_counts_per_time.max()]
            else:
                max_oper_counts = [1]

        return max(max_oper_counts) if max_oper_counts else 1

    def _prepare_time_based_samples(self):
        """연속된 시간을 time_window 크기로 분할하여 샘플 생성"""
        self.samples = []

        if self.sample_id_col is not None:
            sample_ids = self.df_encoded[self.sample_id_col].unique()
            for sample_id in sample_ids:
                sample_df = self.df_encoded[self.df_encoded[self.sample_id_col] == sample_id]
                time_samples = self._split_by_time_window(sample_df)
                self.samples.extend(time_samples)
        else:
            time_samples = self._split_by_time_window(self.df_encoded)
            self.samples.extend(time_samples)

    def _split_by_time_window(self, df):
        """DataFrame을 time_window 크기로 분할하여 샘플 생성"""
        if len(df) == 0:
            return []

        # 시간순 정렬
        df = df.sort_values("timekey_hr")
        unique_times = sorted(df["timekey_hr"].unique())
        samples = []

        # 슬라이딩 윈도우 방식으로 샘플 생성
        for start_idx in range(len(unique_times) - self.time_window + 1):
            window_times = unique_times[start_idx : start_idx + self.time_window]

            # 시간 연속성 체크 (선택적)
            if self._check_time_continuity(window_times):
                window_df = df[df["timekey_hr"].isin(window_times)]

                if len(window_df) > 0:
                    sample_data = self._create_windowed_sample(window_df, window_times)
                    if sample_data is not None:
                        samples.append(sample_data)

        return samples

    def _check_time_continuity(self, time_list):
        """시간 연속성 체크 (현재는 모든 윈도우 허용)"""
        return True

    def _create_windowed_sample(self, df, window_times):
        """
        시간 윈도우 기반 단일 샘플 데이터 구성
        3D 구조: [time_window, max_oper_per_hour, features] 생성
        """
        try:
            grouped = df.groupby("timekey_hr")
            actual_time_steps = len(window_times)

            continuous_data = []
            categorical_data = []
            target_data = []
            hour_oper_counts = []
            timekey_hr_info = []
            oper_id_info = []

            # 각 시간 단위별로 데이터 처리
            for time_key in window_times:
                if time_key in grouped.groups:
                    time_data = grouped.get_group(time_key)
                    time_data = time_data.sort_values("oper_id")

                    # 연속형 데이터
                    time_continuous = (
                        time_data[self.continuous_columns].values
                        if self.continuous_columns
                        else np.empty((len(time_data), 0))
                    )

                    # 범주형 데이터 - 안전성 검증 추가
                    if self.categorical_columns:
                        time_categorical = time_data[self.categorical_columns].values
                        # 범주형 데이터의 값이 유효 범위 내에 있는지 확인
                        for col_idx, col_name in enumerate(self.categorical_columns):
                            vocab_size = len(self.categorical_processor.label_encoders[col_name].classes_)
                            time_categorical[:, col_idx] = np.clip(
                                time_categorical[:, col_idx], 0, vocab_size - 1
                            )
                    else:
                        time_categorical = np.empty((len(time_data), 0))

                    # 타겟 데이터
                    time_target = time_data[self.target_column].values

                    # oper_id 정보
                    time_oper_ids = time_data["oper_id"].values

                    # 해당 시간의 데이터 개수
                    hour_oper_counts.append(len(time_data))
                    timekey_hr_info.append(time_key)
                    oper_id_info.append(time_oper_ids)
                else:
                    # 해당 시간에 데이터가 없는 경우 빈 배열 생성
                    time_continuous = (
                        np.empty((0, len(self.continuous_columns)))
                        if self.continuous_columns
                        else np.empty((0, 0))
                    )
                    time_categorical = (
                        np.empty((0, len(self.categorical_columns)), dtype=np.int64)
                        if self.categorical_columns
                        else np.empty((0, 0))
                    )
                    time_target = np.empty(0)

                    hour_oper_counts.append(0)
                    timekey_hr_info.append(time_key)
                    oper_id_info.append(np.array([], dtype=int))

                continuous_data.append(time_continuous)
                categorical_data.append(time_categorical)
                target_data.append(time_target)

            # time_window 크기만큼 패딩 처리
            while len(hour_oper_counts) < self.time_window:
                hour_oper_counts.append(0)
                continuous_data.append(
                    np.empty((0, len(self.continuous_columns)))
                    if self.continuous_columns
                    else np.empty((0, 0))
                )
                categorical_data.append(
                    np.empty((0, len(self.categorical_columns)), dtype=np.int64)
                    if self.categorical_columns
                    else np.empty((0, 0))
                )
                target_data.append(np.empty(0))
                timekey_hr_info.append(None)
                oper_id_info.append(np.array([], dtype=int))

            # 3D 구조로 반환
            return {
                "continuous_data": continuous_data,  # List[np.array]
                "categorical_data": categorical_data,  # List[np.array]
                "target_data": target_data,  # List[np.array]
                "hour_oper_counts": hour_oper_counts,  # List[int]
                "max_oper_per_hour": self.global_max_oper,  # int
                "actual_time_steps": actual_time_steps,  # int
                "window_times": list(window_times),  # List[int]
                "timekey_hr_info": timekey_hr_info,  # List
                "oper_id_info": oper_id_info,  # List[np.array]
            }

        except Exception as e:
            logger.error(f"샘플 생성 중 오류 발생: {e}")
            return None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """3D 구조를 유지한 샘플 반환"""
        sample = self.samples[idx]
        return {
            "continuous_data": sample["continuous_data"],  # List[numpy배열]
            "categorical_data": sample["categorical_data"],  # List[numpy배열]
            "target_data": sample["target_data"],  # List[numpy배열]
            "hour_oper_counts": sample["hour_oper_counts"],  # List[정수]
            "max_oper_per_hour": sample["max_oper_per_hour"],  # 정수
            "actual_time_steps": sample["actual_time_steps"],  # 정수
            "window_times": sample["window_times"],  # List[정수]
            "timekey_hr_info": sample["timekey_hr_info"],  # List
            "oper_id_info": sample["oper_id_info"],  # List[numpy배열]
        }

### 배치 패딩 함수

In [ ]:
def pad_batch_samples(
    batch_samples,
    continuous_padding_value=0.0,
    categorical_padding_value=0,  # 임베딩의 padding_idx와 일치
    target_padding_value=0.0,
    global_max_oper=None,
):
    """
    배치 내 샘플들을 동일한 크기로 패딩
    최종 출력: [batch_size, time_window, max_oper_per_hour, features]
    """
    batch_size = len(batch_samples)
    time_window = len(batch_samples[0]["continuous_data"])

    # global_max_oper가 주어지면 그것을 사용, 아니면 배치 내 최대값 사용
    if global_max_oper is not None:
        batch_max_oper = global_max_oper
    else:
        batch_max_oper = max(sample["max_oper_per_hour"] for sample in batch_samples)

    first_sample = batch_samples[0]

    # 차원 정보 추출
    continuous_dim = 0
    for hour_data in first_sample["continuous_data"]:
        if len(hour_data) > 0:
            continuous_dim = len(hour_data[0])
            break

    categorical_dim = 0
    for hour_data in first_sample["categorical_data"]:
        if len(hour_data) > 0:
            categorical_dim = len(hour_data[0])
            break

    # 4D 배치 데이터 초기화: [batch_size, time_window, max_oper, features]
    batch_continuous = (
        np.full(
            (batch_size, time_window, batch_max_oper, continuous_dim),
            continuous_padding_value,
            dtype=np.float32,
        )
        if continuous_dim > 0
        else np.empty((batch_size, time_window, batch_max_oper, 0))
    )

    batch_categorical = (
        np.full(
            (batch_size, time_window, batch_max_oper, categorical_dim),
            categorical_padding_value,  # 0으로 설정됨
            dtype=np.int64,
        )
        if categorical_dim > 0
        else np.empty((batch_size, time_window, batch_max_oper, 0))
    )

    batch_targets = np.full(
        (batch_size, time_window, batch_max_oper),
        target_padding_value,
        dtype=np.float32,
    )

    # 마스크: True = 패딩된 위치
    batch_masks = np.ones((batch_size, time_window, batch_max_oper), dtype=bool)
    batch_time_masks = np.zeros((batch_size, time_window), dtype=bool)

    batch_hour_counts = []
    batch_actual_time_steps = []
    batch_window_times = []
    batch_timekey_hr_info = []
    batch_oper_id_info = []

    for batch_idx, sample in enumerate(batch_samples):
        continuous_data = sample["continuous_data"]
        categorical_data = sample["categorical_data"]
        target_data = sample["target_data"]
        hour_counts = sample["hour_oper_counts"]
        actual_time_steps = sample["actual_time_steps"]
        window_times = sample["window_times"]
        timekey_hr_info = sample["timekey_hr_info"]
        oper_id_info = sample["oper_id_info"]

        batch_hour_counts.append(hour_counts)
        batch_actual_time_steps.append(actual_time_steps)
        batch_window_times.append(window_times)
        batch_timekey_hr_info.append(timekey_hr_info)
        batch_oper_id_info.append(oper_id_info)

        # 시간 마스크 설정
        batch_time_masks[batch_idx, :actual_time_steps] = False
        batch_time_masks[batch_idx, actual_time_steps:] = True

        # 각 시간 슬롯별로 데이터 채우기
        for time_idx in range(time_window):
            time_oper_count = hour_counts[time_idx]

            if time_oper_count > 0:
                actual_count = min(time_oper_count, batch_max_oper)

                if continuous_dim > 0:
                    batch_continuous[batch_idx, time_idx, :actual_count] = (
                        continuous_data[time_idx][:actual_count]
                    )

                if categorical_dim > 0:
                    # 범주형 데이터 안전성 검증 및 클램핑
                    cat_data = categorical_data[time_idx][:actual_count]
                    cat_data = np.clip(cat_data, 0, 10000)  # 임시 상한값
                    batch_categorical[batch_idx, time_idx, :actual_count] = cat_data

                batch_targets[batch_idx, time_idx, :actual_count] = target_data[time_idx][:actual_count]

                # 실제 데이터가 있는 위치는 마스크 해제
                batch_masks[batch_idx, time_idx, :actual_count] = False

    # PyTorch 텐서로 변환
    result = {
        "continuous_data": torch.tensor(batch_continuous),
        "categorical_data": torch.tensor(batch_categorical),
        "targets": torch.tensor(batch_targets),
        "masks": torch.tensor(batch_masks),
        "time_masks": torch.tensor(batch_time_masks),
        "hour_counts": batch_hour_counts,
        "actual_time_steps": batch_actual_time_steps,
        "window_times": batch_window_times,
        "max_oper_per_hour": batch_max_oper,
        "timekey_hr_info": batch_timekey_hr_info,
        "oper_id_info": batch_oper_id_info,
    }

    return result


class TimeSeriesOperCollate:
    """배치 패딩을 위한 Collate 클래스"""

    def __init__(
        self,
        continuous_padding_value=0.0,
        categorical_padding_value=0,
        target_padding_value=0.0,
        global_max_oper=None,
    ):
        self.continuous_padding_value = continuous_padding_value
        self.categorical_padding_value = categorical_padding_value
        self.target_padding_value = target_padding_value
        self.global_max_oper = global_max_oper

    def __call__(self, batch):
        return pad_batch_samples(
            batch,
            continuous_padding_value=self.continuous_padding_value,
            categorical_padding_value=self.categorical_padding_value,
            target_padding_value=self.target_padding_value,
            global_max_oper=self.global_max_oper,
        )

### DataLoader 생성 함수

In [ ]:
def create_dataloaders(full_dataset, dataset_config: dict):
    """전체 Dataset으로부터 훈련/검증/테스트 DataLoader 생성"""

    batch_size = dataset_config.get("batch_size", 32)
    train_ratio = dataset_config.get("train_ratio", 0.7)
    val_ratio = dataset_config.get("val_ratio", 0.15)
    test_ratio = dataset_config.get("test_ratio", 0.15)
    num_workers = dataset_config.get("num_workers", 4)

    # 전체 데이터셋의 global_max_oper 값을 미리 저장
    global_max_oper = full_dataset.global_max_oper
    categorical_processor = full_dataset.categorical_processor

    # 데이터셋 분할
    train_dataset, val_dataset, test_dataset = split_dataset_by_ratio(
        full_dataset, train_ratio, val_ratio, test_ratio
    )

    # 분할된 각 데이터셋에 대해 새로운 TimeSeriesOperDataset 생성
    train_df = (
        train_dataset.dataset.df.iloc[train_dataset.indices]
        if hasattr(train_dataset, "indices")
        else train_dataset.df
    )
    val_df = (
        val_dataset.dataset.df.iloc[val_dataset.indices]
        if hasattr(val_dataset, "indices")
        else val_dataset.df
    )
    test_df = (
        test_dataset.dataset.df.iloc[test_dataset.indices]
        if hasattr(test_dataset, "indices")
        else test_dataset.df
    )

    # 새로운 데이터셋 생성 (global_max_oper와 categorical_processor 공유)
    train_dataset_new = TimeSeriesOperDataset(
        df=train_df,
        categorical_columns=full_dataset.categorical_columns,
        continuous_columns=full_dataset.continuous_columns,
        target_column=full_dataset.target_column,
        categorical_processor=categorical_processor,
        time_window=full_dataset.time_window,
        embedding_dim=full_dataset.embedding_dim,
        sample_id_col=full_dataset.sample_id_col,
        global_max_oper=global_max_oper,
    )

    val_dataset_new = TimeSeriesOperDataset(
        df=val_df,
        categorical_columns=full_dataset.categorical_columns,
        continuous_columns=full_dataset.continuous_columns,
        target_column=full_dataset.target_column,
        categorical_processor=categorical_processor,
        time_window=full_dataset.time_window,
        embedding_dim=full_dataset.embedding_dim,
        sample_id_col=full_dataset.sample_id_col,
        global_max_oper=global_max_oper,
    )

    test_dataset_new = TimeSeriesOperDataset(
        df=test_df,
        categorical_columns=full_dataset.categorical_columns,
        continuous_columns=full_dataset.continuous_columns,
        target_column=full_dataset.target_column,
        categorical_processor=categorical_processor,
        time_window=full_dataset.time_window,
        embedding_dim=full_dataset.embedding_dim,
        sample_id_col=full_dataset.sample_id_col,
        global_max_oper=global_max_oper,
    )

    # 패딩값을 0으로 통일 (임베딩 레이어와 일치)
    collate_fn = TimeSeriesOperCollate(
        continuous_padding_value=dataset_config.get("continuous_padding_value", 0.0),
        categorical_padding_value=0,  # 임베딩 레이어의 padding_idx와 일치
        target_padding_value=dataset_config.get("target_padding_value", 0.0),
        global_max_oper=global_max_oper,
    )

    train_dataloader = DataLoader(
        train_dataset_new,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=num_workers,
        pin_memory=True,
    )

    val_dataloader = DataLoader(
        val_dataset_new,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_dataloader = DataLoader(
        test_dataset_new,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=num_workers,
        pin_memory=True,
    )

    logger.info(f"DataLoader 생성 완료:")
    logger.info(f"  - 배치 크기: {batch_size}")
    logger.info(f"  - 전체 데이터셋 global_max_oper: {global_max_oper}")
    logger.info(f"  - 범주형 패딩값: 0 (임베딩 레이어와 일치)")
    logger.info(f"  - 훈련 배치 수: {len(train_dataloader)}")
    logger.info(f"  - 검증 배치 수: {len(val_dataloader)}")
    logger.info(f"  - 테스트 배치 수: {len(test_dataloader)}")

    return train_dataloader, val_dataloader, test_dataloader


## 🏗️ 모델 구조

### 1. 범주형 임베딩 레이어

In [ ]:
class CategoricalEmbeddingLayer(nn.Module):
    """범주형 변수들을 위한 임베딩 레이어"""

    def __init__(self, categorical_processor, embedding_dim=8, padding_value=0):
        super().__init__()
        self.categorical_processor = categorical_processor
        self.embedding_dim = embedding_dim
        self.padding_value = padding_value
        self.embeddings = nn.ModuleList()
        self.vocab_sizes = []

        for col_name, encoder in categorical_processor.label_encoders.items():
            # +1은 패딩을 위한 것 (인덱스 0은 패딩값으로 예약)
            vocab_size = len(encoder.classes_) + 1
            self.vocab_sizes.append(vocab_size)
            emb_layer = nn.Embedding(
                num_embeddings=vocab_size,
                embedding_dim=embedding_dim,
                padding_idx=0,  # 패딩 인덱스는 항상 0
            )
            self.embeddings.append(emb_layer)

        logger.info(f"임베딩 레이어 생성:")
        for i, (col_name, vocab_size) in enumerate(
            zip(categorical_processor.label_encoders.keys(), self.vocab_sizes)
        ):
            logger.info(f"  - {col_name}: vocab_size={vocab_size}, embedding_dim={embedding_dim}")

    def forward(self, categorical_data):
        """
        Input: [batch_size, time_window, max_oper, num_categorical_features]
        Output: [batch_size, time_window, max_oper, num_categorical_features * embedding_dim]
        """
        batch_size, time_window, max_oper, num_features = categorical_data.shape
        embedded_features = []

        for feat_idx in range(num_features):
            feat_data = categorical_data[:, :, :, feat_idx]

            # 인덱스 범위 검증 및 클램핑
            vocab_size = self.vocab_sizes[feat_idx]
            feat_data = torch.clamp(feat_data, 0, vocab_size - 1)

            embedded_feat = self.embeddings[feat_idx](feat_data)
            embedded_features.append(embedded_feat)

        if embedded_features:
            embedded_output = torch.cat(embedded_features, dim=-1)
        else:
            embedded_output = torch.empty(
                batch_size, time_window, max_oper, 0, device=categorical_data.device
            )

        return embedded_output

### 2. 특성 융합 레이어

In [ ]:
class FeatureFusionLayer(nn.Module):
    """범주형 임베딩과 연속형 변수를 결합하고 1차 Linear 변환"""

    def __init__(
        self, continuous_dim: int, categorical_embed_dim: int, dropout: float = 0.1
    ):
        super().__init__()
        self.total_feature_dim = continuous_dim + categorical_embed_dim
        self.feature_projection = nn.Sequential(
            nn.Linear(self.total_feature_dim, 1), 
            nn.Dropout(dropout)
        )

    def forward(self, continuous_data, categorical_embedded):
        """
        Input: 
        - continuous_data: [batch_size, time_window, max_oper, continuous_dim]
        - categorical_embedded: [batch_size, time_window, max_oper, categorical_embed_dim]
        Output: [batch_size, time_window, max_oper]
        """
        if categorical_embedded.shape[-1] > 0:
            combined_features = torch.cat([continuous_data, categorical_embedded], dim=-1)
        else:
            combined_features = continuous_data

        # [batch_size, time_window, max_oper, total_dim] -> [batch_size, time_window, max_oper]
        projected = self.feature_projection(combined_features).squeeze(-1)
        return projected

### 3. Transformer 백본

In [ ]:
class TransformerBackbone(nn.Module):
    """Transformer Encoder 백본"""

    def __init__(
        self,
        max_oper_per_hour: int,
        time_window: int = 24,
        num_layers: int = 6,
        num_heads: int = 8,
        hidden_dim: int = 512,
        feedforward_dim: int = 2048,
        dropout: float = 0.1,
        activation: str = "relu",
    ):
        super().__init__()
        self.hidden_dim = hidden_dim

        # 입력 투영: 1D -> hidden_dim
        self.input_projection = nn.Linear(1, hidden_dim)
        
        # 위치 인코딩: 시간축과 공정축 모두 고려
        self.time_pos_encoding = nn.Parameter(torch.randn(1, time_window, 1, hidden_dim))
        self.oper_pos_encoding = nn.Parameter(torch.randn(1, 1, max_oper_per_hour, hidden_dim))

        # Transformer 인코더
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            activation=activation,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 출력 투영: hidden_dim -> 1
        self.output_projection = nn.Linear(hidden_dim, 1)

    def forward(self, x, masks=None, time_masks=None):
        """
        Input: x [batch_size, time_window, max_oper]
        Output: [batch_size, time_window, max_oper]
        """
        batch_size, time_window, max_oper = x.shape

        # 1D -> hidden_dim 변환
        x_expanded = x.unsqueeze(-1)  # [batch_size, time_window, max_oper, 1]
        x_projected = self.input_projection(x_expanded)  # [batch_size, time_window, max_oper, hidden_dim]

        # 위치 인코딩 추가
        x_projected = x_projected + self.time_pos_encoding[:, :time_window, :, :]
        x_projected = x_projected + self.oper_pos_encoding[:, :, :max_oper, :]

        # Transformer 입력을 위해 2D로 reshape: [batch_size, time_window * max_oper, hidden_dim]
        x_reshaped = x_projected.view(batch_size, time_window * max_oper, self.hidden_dim)

        # 마스크 처리
        src_key_padding_mask = None
        if masks is not None:
            src_key_padding_mask = masks.view(batch_size, time_window * max_oper)

        # Transformer forward
        transformer_output = self.transformer_encoder(
            x_reshaped, src_key_padding_mask=src_key_padding_mask
        )
        
        # 다시 3D로 reshape: [batch_size, time_window, max_oper, hidden_dim]
        transformer_output = transformer_output.view(batch_size, time_window, max_oper, self.hidden_dim)

        # 최종 출력: [batch_size, time_window, max_oper]
        output = self.output_projection(transformer_output).squeeze(-1)

        # 마스킹된 위치는 0으로 설정
        if masks is not None:
            output = output.masked_fill(masks, 0.0)

        return output


### 4. LSTM 백본 (선택사항)

In [ ]:
class LSTMBackbone(nn.Module):
    """LSTM 백본 - Transformer 대안"""

    def __init__(
        self,
        max_oper_per_hour: int,
        time_window: int = 24,
        num_layers: int = 2,
        hidden_dim: int = 128,
        dropout: float = 0.1,
        bidirectional: bool = True,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.input_projection = nn.Linear(1, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
            bidirectional=bidirectional,
        )

        lstm_output_dim = hidden_dim * (2 if bidirectional else 1)
        self.output_projection = nn.Sequential(
            nn.Linear(lstm_output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, masks=None, time_masks=None):
        """
        Input: x [batch_size, time_window, max_oper]
        Output: [batch_size, time_window, max_oper]
        """
        batch_size, time_window, max_oper = x.shape

        x_expanded = x.unsqueeze(-1)
        x_projected = self.input_projection(x_expanded)

        # 각 공정별로 시계열 처리: [batch_size * max_oper, time_window, hidden_dim]
        x_reshaped = x_projected.transpose(1, 2).contiguous()
        x_reshaped = x_reshaped.view(-1, time_window, self.hidden_dim)

        lstm_output, _ = self.lstm(x_reshaped)
        lstm_output = lstm_output.view(batch_size, max_oper, time_window, -1)
        lstm_output = lstm_output.transpose(1, 2)

        output = self.output_projection(lstm_output).squeeze(-1)

        if masks is not None:
            output = output.masked_fill(masks, 0.0)

        return output


### 5. 완전한 TimeOperTransformer 모델

In [ ]:
class TimeSeriesOperModel(nn.Module):
    """완전한 시계열 공정 데이터 모델 (Feature Fusion + Backbone)"""

    def __init__(
        self,
        categorical_processor,
        continuous_dim: int,
        max_oper_per_hour: int,
        config: dict,
        time_window: int = 24,
    ):
        super().__init__()

        self.backbone_type = config.get("backbone_type", "transformer")
        embedding_dim = config.get("embedding_dim", 8)
        dropout = config.get("dropout", 0.1)

        # 범주형 임베딩 레이어
        self.categorical_embedding = CategoricalEmbeddingLayer(
            categorical_processor, embedding_dim, padding_value=0
        )

        num_categorical_features = len(categorical_processor.label_encoders)
        categorical_embed_dim = num_categorical_features * embedding_dim

        # 특성 융합 레이어
        self.feature_fusion = FeatureFusionLayer(
            continuous_dim=continuous_dim,
            categorical_embed_dim=categorical_embed_dim,
            dropout=dropout,
        )

        # 백본 모델 선택
        if self.backbone_type == "transformer":
            transformer_config = config.get("transformer", {})
            self.backbone = TransformerBackbone(
                max_oper_per_hour=max_oper_per_hour,
                time_window=time_window,
                **transformer_config,
            )
        elif self.backbone_type == "lstm":
            lstm_config = config.get("lstm", {})
            self.backbone = LSTMBackbone(
                max_oper_per_hour=max_oper_per_hour,
                time_window=time_window,
                **lstm_config,
            )
        else:
            raise ValueError(f"Unknown backbone_type: {self.backbone_type}")

        logger.info(f"TimeSeriesOperModel 초기화 완료:")
        logger.info(f"  - Backbone: {self.backbone_type}")
        logger.info(f"  - 임베딩 차원: {embedding_dim}")
        logger.info(f"  - 총 파라미터 수: {sum(p.numel() for p in self.parameters()):,}")

    def forward(self, continuous_data, categorical_data, masks=None, time_masks=None):
        """
        Input:
        - continuous_data: [batch_size, time_window, max_oper, continuous_dim]
        - categorical_data: [batch_size, time_window, max_oper, categorical_dim]
        - masks: [batch_size, time_window, max_oper] (True = 패딩)
        - time_masks: [batch_size, time_window] (True = 패딩)
        
        Output: [batch_size, time_window, max_oper]
        """
        batch_size, time_window, max_oper, _ = continuous_data.shape

        # 범주형 데이터 임베딩
        if categorical_data.shape[-1] > 0:
            # 범주형 데이터의 값 범위 체크 및 클램핑
            categorical_data = torch.clamp(categorical_data, 0, 10000)  # 임시 상한값
            categorical_embedded = self.categorical_embedding(categorical_data)
        else:
            categorical_embedded = torch.empty(
                batch_size, time_window, max_oper, 0, device=continuous_data.device
            )

        # 특성 융합: [batch_size, time_window, max_oper]
        fused_output = self.feature_fusion(continuous_data, categorical_embedded)
        
        # 백본 모델을 통한 예측
        predictions = self.backbone(fused_output, masks, time_masks)

        return predictions


def create_model_from_dataloader(dataloader, model_config: dict):
    """DataLoader에서 정보를 추출하여 모델 생성"""

    for batch in dataloader:
        continuous_dim = batch["continuous_data"].shape[-1]
        max_oper_per_hour = batch["max_oper_per_hour"]
        time_window = batch["continuous_data"].shape[1]

        # Dataset으로부터 categorical_processor 가져오기
        if hasattr(dataloader.dataset, "categorical_processor"):
            categorical_processor = dataloader.dataset.categorical_processor
        else:
            # Subset인 경우
            categorical_processor = dataloader.dataset.dataset.categorical_processor

        logger.info(f"DataLoader로부터 추출된 정보:")
        logger.info(f"  - continuous_dim: {continuous_dim}")
        logger.info(f"  - max_oper_per_hour: {max_oper_per_hour}")
        logger.info(f"  - time_window: {time_window}")
        logger.info(f"  - 범주형 변수 개수: {len(categorical_processor.label_encoders)}")

        # 범주형 변수별 vocab_size 출력
        for col_name, encoder in categorical_processor.label_encoders.items():
            vocab_size = len(encoder.classes_) + 1
            logger.info(f"  - {col_name}: vocab_size={vocab_size}")

        break

    model = TimeSeriesOperModel(
        categorical_processor=categorical_processor,
        continuous_dim=continuous_dim,
        max_oper_per_hour=max_oper_per_hour,
        config=model_config,
        time_window=time_window,
    )

    return model


## 🚂 훈련 관련 함수들

### 1. 마스크 기반 손실 함수

In [ ]:
class MaskedMSELoss(nn.Module):
    """마스크를 고려한 MSE Loss"""

    def __init__(self, reduction="mean"):
        super().__init__()
        self.reduction = reduction

    def forward(self, predictions, targets, masks):
        """
        predictions: [batch_size, time_window, max_oper]
        targets: [batch_size, time_window, max_oper]
        masks: [batch_size, time_window, max_oper] (True = 패딩)
        """
        valid_mask = ~masks  # 실제 데이터 위치

        if valid_mask.sum() == 0:
            return torch.tensor(0.0, device=predictions.device, requires_grad=True)

        valid_predictions = predictions[valid_mask]
        valid_targets = targets[valid_mask]

        mse_loss = nn.functional.mse_loss(
            valid_predictions, valid_targets, reduction=self.reduction
        )
        return mse_loss

### 2. 메트릭 계산 함수

In [ ]:
def compute_metrics(predictions, targets, masks, epsilon=1e-8):
    """성능 메트릭 계산 (MSE, RMSE, MAE, MAPE)"""
    valid_mask = ~masks
    valid_predictions = predictions[valid_mask]
    valid_targets = targets[valid_mask]

    if len(valid_predictions) == 0:
        return {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}

    mse = torch.mean((valid_predictions - valid_targets) ** 2).item()
    rmse = torch.sqrt(torch.mean((valid_predictions - valid_targets) ** 2)).item()
    mae = torch.mean(torch.abs(valid_predictions - valid_targets)).item()

    # MAPE 계산 (0으로 나누기 방지)
    abs_targets = torch.abs(valid_targets)
    abs_errors = torch.abs(valid_predictions - valid_targets)
    safe_targets = torch.clamp(abs_targets, min=epsilon)
    mape = torch.mean(abs_errors / safe_targets * 100).item()

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "valid_count": len(valid_predictions),
    }

### 3. 훈련 에폭 함수

In [ ]:
def train_epoch(model, train_dataloader, criterion, optimizer, device, epoch, log_frequency=50):
    """한 에폭 훈련"""
    model.train()
    total_loss = 0.0
    total_metrics = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}

    # tqdm 진행바 설정
    pbar = tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        desc=f"Epoch {epoch} [Train]",
        leave=False,
        ncols=100,
    )

    for batch_idx, batch in pbar:
        continuous_data = batch["continuous_data"].to(device)
        categorical_data = batch["categorical_data"].to(device)
        targets = batch["targets"].to(device)
        masks = batch["masks"].to(device)
        time_masks = batch["time_masks"].to(device)

        # Forward pass
        predictions = model(continuous_data, categorical_data, masks, time_masks)
        loss = criterion(predictions, targets, masks)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 메트릭 계산
        with torch.no_grad():
            batch_metrics = compute_metrics(predictions, targets, masks)

        total_loss += loss.item()
        for key in ["mse", "rmse", "mae", "mape"]:
            total_metrics[key] += batch_metrics[key]
        total_metrics["valid_count"] += batch_metrics["valid_count"]

        # 진행바 업데이트
        current_avg_loss = total_loss / (batch_idx + 1)
        current_avg_mape = total_metrics["mape"] / (batch_idx + 1)
        pbar.set_postfix({
            "Loss": f"{current_avg_loss:.4f}",
            "MAPE": f"{current_avg_mape:.2f}%",
            "Batch_Loss": f"{loss.item():.4f}",
        })

        # 주기적 로깅
        if batch_idx % log_frequency == 0 and batch_idx > 0:
            logger.info(
                f"Epoch {epoch}, Batch {batch_idx}/{len(train_dataloader)}, "
                f'Loss: {loss.item():.4f}, MAPE: {batch_metrics["mape"]:.2f}%'
            )

    pbar.close()

    avg_loss = total_loss / len(train_dataloader)
    for key in ["mse", "rmse", "mae", "mape"]:
        total_metrics[key] = total_metrics[key] / len(train_dataloader)

    return avg_loss, total_metrics

### 4. 검증 에폭 함수

In [ ]:
def validate_epoch(model, val_dataloader, criterion, device, epoch=None):
    """검증 에폭"""
    model.eval()
    total_loss = 0.0
    total_metrics = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}

    desc = f"Epoch {epoch} [Val]" if epoch is not None else "Validation"
    pbar = tqdm(val_dataloader, desc=desc, leave=False, ncols=100)

    with torch.no_grad():
        for batch_idx, batch in enumerate(pbar):
            continuous_data = batch["continuous_data"].to(device)
            categorical_data = batch["categorical_data"].to(device)
            targets = batch["targets"].to(device)
            masks = batch["masks"].to(device)
            time_masks = batch["time_masks"].to(device)

            predictions = model(continuous_data, categorical_data, masks, time_masks)
            loss = criterion(predictions, targets, masks)
            batch_metrics = compute_metrics(predictions, targets, masks)

            total_loss += loss.item()
            for key in ["mse", "rmse", "mae", "mape"]:
                total_metrics[key] += batch_metrics[key]
            total_metrics["valid_count"] += batch_metrics["valid_count"]

            # 진행바 업데이트
            current_avg_loss = total_loss / (batch_idx + 1)
            current_avg_mape = total_metrics["mape"] / (batch_idx + 1)
            pbar.set_postfix({
                "Loss": f"{current_avg_loss:.4f}", 
                "MAPE": f"{current_avg_mape:.2f}%"
            })

    pbar.close()

    avg_loss = total_loss / len(val_dataloader)
    for key in ["mse", "rmse", "mae", "mape"]:
        total_metrics[key] = total_metrics[key] / len(val_dataloader)

    return avg_loss, total_metrics

### 5. 메인 훈련 루프

In [ ]:
def train_model(model, train_dataloader, val_dataloader, training_config: dict, device: str):
    """모델 훈련 메인 함수"""
    num_epochs = training_config.get("num_epochs", 100)
    learning_rate = training_config.get("learning_rate", 1e-3)
    patience = training_config.get("patience", 10)
    save_path = training_config.get("save_path", "best_model.pth")
    log_frequency = training_config.get("log_frequency", 50)

    # 훈련 구성 요소
    criterion = MaskedMSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=patience // 2, verbose=True
    )

    model = model.to(device)

    train_history = {"loss": [], "mse": [], "rmse": [], "mae": [], "mape": []}
    val_history = {"loss": [], "mse": [], "rmse": [], "mae": [], "mape": []}

    best_val_loss = float("inf")
    patience_counter = 0

    logger.info(f"모델 훈련 시작: {num_epochs}개 에폭, 학습률 {learning_rate}")

    # 전체 에폭에 대한 tqdm 진행바
    epoch_pbar = tqdm(
        range(1, num_epochs + 1),
        desc="Training Progress",
        ncols=120,
    )

    for epoch in epoch_pbar:
        train_loss, train_metrics = train_epoch(
            model, train_dataloader, criterion, optimizer, device, epoch, log_frequency
        )
        val_loss, val_metrics = validate_epoch(
            model, val_dataloader, criterion, device, epoch
        )

        scheduler.step(val_loss)

        train_history["loss"].append(train_loss)
        for key in ["mse", "rmse", "mae", "mape"]:
            train_history[key].append(train_metrics[key])
            val_history[key].append(val_metrics[key])
        val_history["loss"].append(val_loss)

        # 에폭별 결과 로깅
        logger.info(
            f"Epoch {epoch:3d}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
            f'Train MAPE={train_metrics["mape"]:.2f}%, Val MAPE={val_metrics["mape"]:.2f}%'
        )

        # 전체 에폭 진행바 업데이트
        epoch_pbar.set_postfix({
            "T_Loss": f"{train_loss:.4f}",
            "V_Loss": f"{val_loss:.4f}",
            "T_MAPE": f'{train_metrics["mape"]:.2f}%',
            "V_MAPE": f'{val_metrics["mape"]:.2f}%',
            "Best": f"{best_val_loss:.4f}",
            "Patience": f"{patience_counter}/{patience}",
        })

        # 최고 모델 저장
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_metrics": val_metrics,
                "train_metrics": train_metrics,
            }, save_path)
            logger.info(f"  → Best model saved! (Val Loss: {val_loss:.4f})")
        else:
            patience_counter += 1

        # 조기 종료
        if patience_counter >= patience:
            logger.info(f"Early stopping at epoch {epoch}")
            epoch_pbar.close()
            break

    if patience_counter < patience:
        epoch_pbar.close()

    return {
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
    }

### 6. 구조화된 테스트 함수

In [ ]:
def test_model_with_structure(model, test_dataloader, device, model_path):
    """구조화된 테스트 (timekey_hr, oper_id 복원)"""
    logger.info(f"저장된 모델 로드: {model_path}")
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    model = model.to(device)
    model.eval()

    criterion = MaskedMSELoss()
    structured_predictions = []
    valid_predictions_list = []
    valid_targets_list = []
    total_loss = 0.0

    logger.info(f"구조화된 테스트 시작")

    # 테스트용 tqdm 진행바
    pbar = tqdm(
        enumerate(test_dataloader),
        total=len(test_dataloader),
        desc="Testing",
        ncols=100,
    )

    with torch.no_grad():
        for batch_idx, batch in pbar:
            continuous_data = batch["continuous_data"].to(device)
            categorical_data = batch["categorical_data"].to(device)
            targets = batch["targets"].to(device)
            masks = batch["masks"].to(device)
            time_masks = batch["time_masks"].to(device)

            timekey_hr_info_batch = batch["timekey_hr_info"]
            oper_id_info_batch = batch["oper_id_info"]

            predictions = model(continuous_data, categorical_data, masks, time_masks)
            loss = criterion(predictions, targets, masks)
            total_loss += loss.item()

            predictions_cpu = predictions.cpu()
            targets_cpu = targets.cpu()
            masks_cpu = masks.cpu()

            batch_size = predictions_cpu.shape[0]
            for sample_idx in range(batch_size):
                sample_predictions = predictions_cpu[sample_idx]
                sample_targets = targets_cpu[sample_idx]
                sample_masks = masks_cpu[sample_idx]

                sample_timekey_hr_info = timekey_hr_info_batch[sample_idx]
                sample_oper_id_info = oper_id_info_batch[sample_idx]

                for time_idx, (timekey_hr, oper_ids_array) in enumerate(
                    zip(sample_timekey_hr_info, sample_oper_id_info)
                ):
                    if timekey_hr is not None and len(oper_ids_array) > 0:
                        for oper_idx_in_array, actual_oper_id in enumerate(oper_ids_array):
                            if not sample_masks[time_idx, oper_idx_in_array]:
                                pred_val = sample_predictions[time_idx, oper_idx_in_array].item()
                                target_val = sample_targets[time_idx, oper_idx_in_array].item()

                                structured_predictions.append({
                                    "timekey_hr": timekey_hr,
                                    "oper_id": actual_oper_id,
                                    "predicted": pred_val,
                                    "actual": target_val,
                                })

                                valid_predictions_list.append(pred_val)
                                valid_targets_list.append(target_val)

            # tqdm 진행바 업데이트
            current_avg_loss = total_loss / (batch_idx + 1)
            predictions_count = len(structured_predictions)
            pbar.set_postfix({
                "Loss": f"{current_avg_loss:.4f}",
                "Predictions": f"{predictions_count:,}",
            })

    pbar.close()

    avg_loss = total_loss / len(test_dataloader)

    # 메트릭 계산
    if len(valid_predictions_list) > 0:
        all_valid_predictions = torch.tensor(valid_predictions_list)
        all_valid_targets = torch.tensor(valid_targets_list)

        mse = torch.mean((all_valid_predictions - all_valid_targets) ** 2).item()
        rmse = torch.sqrt(torch.mean((all_valid_predictions - all_valid_targets) ** 2)).item()
        mae = torch.mean(torch.abs(all_valid_predictions - all_valid_targets)).item()

        epsilon = 1e-8
        abs_targets = torch.abs(all_valid_targets)
        abs_errors = torch.abs(all_valid_predictions - all_valid_targets)
        safe_targets = torch.clamp(abs_targets, min=epsilon)
        mape = torch.mean(abs_errors / safe_targets * 100).item()

        metrics = {
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "mape": mape,
            "valid_count": len(valid_predictions_list),
        }
    else:
        metrics = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}

    logger.info(
        f"테스트 결과: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, MAPE={metrics['mape']:.2f}%"
    )
    logger.info(f"구조화된 예측 결과: {len(structured_predictions):,}개")

    return {
        "test_loss": avg_loss,
        "metrics": metrics,
        "structured_predictions": structured_predictions,
    }

## 💾 결과 저장 함수

In [ ]:
def save_results(test_results, output_dir: str):
    """결과 저장"""
    os.makedirs(output_dir, exist_ok=True)

    # 성능 지표 저장
    with open(f"{output_dir}/metrics.json", "w") as f:
        json.dump({
            "test_loss": test_results["test_loss"],
            "metrics": test_results["metrics"],
        }, f, indent=2)

    # 구조화된 예측 결과 저장
    df_predictions = pd.DataFrame(test_results["structured_predictions"])
    df_predictions["error"] = df_predictions["predicted"] - df_predictions["actual"]
    df_predictions["abs_error"] = df_predictions["error"].abs()
    df_predictions["abs_percent_error"] = (
        df_predictions["abs_error"] / df_predictions["actual"].abs().clip(lower=1e-8)
    ) * 100

    df_predictions.to_csv(f"{output_dir}/predictions.csv", index=False)

    logger.info(f"결과 저장 완료: {output_dir}")
    logger.info(f"  - metrics.json: 성능 지표")
    logger.info(f"  - predictions.csv: 예측 결과 ({len(df_predictions):,}개)")

## ⚙️ 설정 파일 구조

### 1. dataset.yaml
```yaml
data_path: "/path/to/your/excel/file.xlsx"
categorical_columns: ["oper_group","days","shift","x1"]
continuous_columns: ["x2","x3","x4",...,"x21"]
sheet_names: ["Data_Set1(사외)", "Data_Set2(사외)"]
additional_drop_columns: ["lot_cd", "oper_area"]
embedding_dim: 8
time_window: 24           # 시간 윈도우 크기
target_column: "y"
batch_size: 64
train_ratio: 0.8
val_ratio: 0.1
test_ratio: 0.1
```

### 2. model.yaml
```yaml
backbone_type: transformer  # transformer 또는 lstm
embedding_dim: 8
dropout: 0.1

transformer: 
  num_layers: 6             # Transformer 레이어 수
  num_heads: 8              # 어텐션 헤드 수
  hidden_dim: 128           # 은닉 차원
  feedforward_dim: 512      # 피드포워드 차원
  dropout: 0.1
  activation: relu

lstm: 
  num_layers: 2             # LSTM 레이어 수
  hidden_dim: 64            # LSTM 은닉 상태 차원
  dropout: 0.1
  bidirectional: true       # 양방향 LSTM
```

### 3. training.yaml
```yaml
num_epochs: 10
learning_rate: 0.001
patience: 10              # 조기 종료 대기 에폭
save_path: 'best_model.pth'
output_dir: 'results'
log_frequency: 50         # 배치 단위 로그 빈도
```


## 🎯 메인 실행 함수

In [ ]:
def run(args):
    """메인 실행 함수"""
    
    # 설정 파일 로드
    logger.info("=" * 60)
    logger.info("시계열 공정 데이터 모델링 파이프라인 시작")
    logger.info("=" * 60)

    dataset_config = load_config(args.dataset_config)
    model_config = load_config(args.model_config)
    training_config = load_config(args.training_config)

    logger.info(f"Dataset 설정 로드: {args.dataset_config}")
    logger.info(f"Model 설정 로드: {args.model_config}")
    logger.info(f"Training 설정 로드: {args.training_config}")

    # 데이터 로드 및 전처리
    data_path = dataset_config["data_path"]
    logger.info(f"Excel 데이터 로드: {data_path}")

    # Excel 파일의 모든 시트 로드 (header=1)
    excel = pd.read_excel(data_path, sheet_name=None, header=1)
    logger.info(f"Excel 시트 개수: {len(excel)}")

    # 지정된 시트들 결합
    sheet_names = dataset_config.get("sheet_names", ["Data_Set1(사외)", "Data_Set2(사외)"])
    logger.info(f"사용할 시트: {sheet_names}")

    total = pd.concat([excel[sheet_name] for sheet_name in sheet_names])
    logger.info(f"시트 결합 후 크기: {total.shape}")

    # 기본 전처리
    if "Unnamed: 0" in total.columns:
        total.drop(columns="Unnamed: 0", inplace=True)
        logger.info("'Unnamed: 0' 컬럼 제거 완료")

    # y값 결측치 제거
    original_size = len(total)
    data = total[~total["y"].isna()]
    logger.info(f"y값 결측치 제거: {original_size:,} → {len(data):,} (-{original_size - len(data):,})")

    # 불필요한 컬럼 제거
    drop_x_features = dataset_config.get("drop_x_features", [f"x{i}" for i in range(22, 49 + 1)])
    if drop_x_features:
        existing_drop_features = [col for col in drop_x_features if col in data.columns]
        if existing_drop_features:
            data = data.drop(columns=existing_drop_features, axis=1)
            logger.info(f"x22-x49 컬럼 제거: {len(existing_drop_features)}개 컬럼 제거")

    additional_drop_columns = dataset_config.get("additional_drop_columns", ["lot_cd", "oper_area"])
    if additional_drop_columns:
        existing_additional_drops = [col for col in additional_drop_columns if col in data.columns]
        if existing_additional_drops:
            data = data.drop(columns=existing_additional_drops, axis=1)
            logger.info(f"추가 컬럼 제거: {existing_additional_drops}")

    data.reset_index(drop=True, inplace=True)

    logger.info(f"최종 전처리 완료:")
    logger.info(f"  - 최종 데이터 크기: {data.shape}")
    logger.info(f"  - 컬럼 목록: {list(data.columns)}")

    # Dataset 생성 (3D 구조)
    logger.info("3D Dataset 생성 중...")
    full_dataset = TimeSeriesOperDataset(
        df=data,
        categorical_columns=dataset_config["categorical_columns"],
        continuous_columns=dataset_config["continuous_columns"],
        target_column=dataset_config["target_column"],
        time_window=dataset_config.get("time_window", 24),
        embedding_dim=model_config.get("embedding_dim", 8),
        sample_id_col=dataset_config.get("sample_id_col", None),
    )

    # DataLoader 생성 (4D 배치 처리)
    logger.info("4D DataLoader 생성 중...")
    train_dataloader, val_dataloader, test_dataloader = create_dataloaders(
        full_dataset, dataset_config
    )

    # 모델 생성
    logger.info("모델 생성 중...")
    model = create_model_from_dataloader(train_dataloader, model_config)

    # GPU 설정
    device = f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu"
    logger.info(f"사용 디바이스: {device}")

    # 훈련
    logger.info("모델 훈련 시작...")
    training_results = train_model(
        model, train_dataloader, val_dataloader, training_config, device
    )

    # 테스트
    logger.info("모델 테스트 시작...")
    model_path = training_config.get("save_path", "best_model.pth")
    test_results = test_model_with_structure(model, test_dataloader, device, model_path)

    # 결과 저장
    output_dir = training_config.get(
        "output_dir", f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
    save_results(test_results, output_dir)

    logger.info("=" * 60)
    logger.info("파이프라인 완료!")
    logger.info(
        f"최종 성능: RMSE={test_results['metrics']['rmse']:.4f}, MAPE={test_results['metrics']['mape']:.2f}%"
    )
    logger.info(f"결과 저장 위치: {output_dir}")
    logger.info(f"모델 저장 위치: {model_path}")
    logger.info("=" * 60)


def parse_args():
    """인자 파싱"""
    import argparse
    
    parser = argparse.ArgumentParser(description="시계열 공정 데이터 모델링 파이프라인")

    parser.add_argument(
        "--dataset_config",
        type=str,
        default="./configs/dataset.yaml",
        help="데이터셋 설정 YAML 파일 경로",
    )
    parser.add_argument(
        "--model_config",
        type=str,
        default="./configs/model.yaml",
        help="모델 설정 YAML 파일 경로",
    )
    parser.add_argument(
        "--training_config",
        type=str,
        default="./configs/training.yaml",
        help="훈련 설정 YAML 파일 경로",
    )
    parser.add_argument(
        "--gpu",
        type=int,
        default=0,
        help="사용할 GPU 번호",
    )

    return parser.parse_args()


# 실행
if __name__ == "__main__":
    args = parse_args()
    run(args)

## 💡 하이퍼파라미터 튜닝 가이드

### 1. 시간 윈도우 크기 조정 (dataset.yaml)

**time_window**:
- 짧은 패턴 학습: `time_window: 12` (12시간)
- 기본값: `time_window: 24` (24시간)
- 긴 패턴 학습: `time_window: 48` (48시간) - 메모리 부족 주의

### 2. 모델 크기 조정 (model.yaml)

**Transformer 설정**:
```yaml
# 작은 모델 (메모리 부족시)
transformer: 
  num_layers: 3
  num_heads: 4
  hidden_dim: 64
  feedforward_dim: 256

# 기본 모델
transformer: 
  num_layers: 6
  num_heads: 8
  hidden_dim: 128
  feedforward_dim: 512

# 큰 모델 (충분한 메모리)
transformer: 
  num_layers: 12
  num_heads: 16
  hidden_dim: 256
  feedforward_dim: 1024
```

**LSTM 설정**:
```yaml
# 작은 모델
lstm: 
  num_layers: 1
  hidden_dim: 32
  bidirectional: false

# 기본 모델
lstm: 
  num_layers: 2
  hidden_dim: 64
  bidirectional: true

# 큰 모델
lstm: 
  num_layers: 4
  hidden_dim: 128
  bidirectional: true
```

### 3. 배치 크기 조정 (dataset.yaml)

**batch_size**:
- GPU 메모리 부족시: `batch_size: 16, 32`
- 기본값: `batch_size: 64`
- 충분한 메모리: `batch_size: 128, 256`

### 4. 성능 최적화 예시

In [ ]:
config_small = {
    "dataset": {
        "time_window": 12,
        "batch_size": 16,
    },
    "model": {
        "backbone_type": "lstm",
        "embedding_dim": 4,
        "lstm": {
            "num_layers": 1,
            "hidden_dim": 32,
            "bidirectional": False
        }
    }
}

# 고성능 설정
config_large = {
    "dataset": {
        "time_window": 24,
        "batch_size": 128,
    },
    "model": {
        "backbone_type": "transformer",
        "embedding_dim": 16,
        "transformer": {
            "num_layers": 12,
            "num_heads": 16,
            "hidden_dim": 256,
            "feedforward_dim": 1024
        }
    }
}

## 📝 실행 예시

```bash
# 기본 설정으로 실행
python main.py

# 특정 GPU 사용
python main.py --gpu 1

# 사용자 정의 설정 파일들 사용
python main.py \
    --dataset_config ./my_configs/dataset.yaml \
    --model_config ./my_configs/model.yaml \
    --training_config ./my_configs/training.yaml